In [2]:
import csv

# make sure the builtin print isn’t shadowed by a string variable
try:
    del print
except NameError:
    pass

def read_csv(file_path):
    with open(file_path, mode='r', encoding='utf-8') as f:
        return list(csv.DictReader(f))

data = read_csv(r'D:\SwedishGeoHistori\SwedishGeoHistory\scraper\krig_events.csv')
# Print first 5 rows to inspect the data structure
for i, row in enumerate(data[:5]):
    print(f"Row {i+1}: {row}")

Row 1: {'year': '500', 'title': 'Krig/konflikt i Sverige 500', 'description': 'Gustav IV Adolf gav inte upp hoppet. Han lyckades, med Preussens hjälp, samla ihop en styrka om 17 500 man, delvis undermåligt tränade. Mot dessa stod den franska armén om 40 000 man. Den 13 juni 1807 började den svenska armén röra på sig, men i början av juli slöt Ryssland och Preussen fred med Frankrike. Den svenska styrkan tvingades därför dra sig tillbaka till Stralsund, varefter man snabbt backade till Rügen. Det franska befälet gick slutligen med på att ge svenskarna fritt avtåg. Fransmänn', 'area': 'Stralsund', 'source_url': 'https://sv.wikipedia.org/wiki/Sveriges_militärhistoria'}
Row 2: {'year': '500', 'title': 'Krig/konflikt i Sverige 500', 'description': 'Göteborgs försvar rustades nu upp i snabb takt, och man utökade exempelvis Göteborgs borgerskaps militärkårer med ytterligare artilleri. Förstärkningstrupper anlände också från flera håll, så att staden snart befann sig i ganska gott försvarsskic

In [3]:
import pandas as pd

# wrap the list of dicts in a DataFrame before using DataFrame methods
df = pd.DataFrame(data)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2626 entries, 0 to 2625
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   year         2626 non-null   object
 1   title        2626 non-null   object
 2   description  2626 non-null   object
 3   area         2626 non-null   object
 4   source_url   2626 non-null   object
dtypes: object(5)
memory usage: 102.7+ KB


In [4]:
def show_counts(df):
    print(df['area'].value_counts())

show_counts(df[df['area'] == "Sverige"])

area
Sverige    65
Name: count, dtype: int64


In [5]:
def get_one_sweden(df):
    print(df[df['area'] == "Sverige"][['year', 'title', 'description', 'area', 'source_url']].iloc[0:6])
    antal_sverige_i_area = df[df['area'] == "Sverige"].count()['area']
    print(f"Antal rader där area är Sverige: {antal_sverige_i_area}")
get_one_sweden(df)

     year               title  \
237  1100  Krig/konflikt 1100   
274  1200  Krig/konflikt 1200   
406  1300  Krig/konflikt 1300   
440  1349  Krig/konflikt 1349   
441  1350  Krig/konflikt 1350   
532  1430  Krig/konflikt 1430   

                                           description     area  \
237  I Sverige inträffade kristendomens slutliga se...  Sverige   
274  I Sverige inträffade kristendomens slutliga se...  Sverige   
406  Senmedeltiden (perioden efter 1350) känneteckn...  Sverige   
440  Senmedeltiden (perioden efter 1350) känneteckn...  Sverige   
441  Senmedeltiden (perioden efter 1350) känneteckn...  Sverige   
532  Kronans ekonomiska svårigheter ledde även till...  Sverige   

                                            source_url  
237  https://www.so-rummet.se/fakta-artiklar/sverig...  
274  https://www.so-rummet.se/fakta-artiklar/sverig...  
406  https://www.so-rummet.se/fakta-artiklar/sverig...  
440  https://www.so-rummet.se/fakta-artiklar/sverig...  
441  https://

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()
api = os.environ.get("GROQ_API_KEY")
if not api:
    raise RuntimeError("GROQ_API_KEY saknas. Lagg den i Ai/.env som GROQ_API_KEY=...")

In [7]:
from groq import Groq

In [8]:
client = Groq(api_key=api)

In [9]:
def ask_groq_rows(df):
    subset = df[df['area'] == "Sverige"]
    antal_sverige_i_df = subset.count()['area']
    print(f"Antal rader där area är Sverige i hela DataFrame: {antal_sverige_i_df}")
    for index, row in subset.iterrows():
        completion = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{
                "role": "system", 
                "content": "Du är en expert på svensk geografi. Svara ENDAST med namnet på den svenska orten eller staden. Om platsen är utanför Sverige, svara 'Utländsk'."
            },
            {
                "role": "user", 
                "content": f"Vilken svensk stad/plats är viktigast här? Svara med max två ord: {row['description']}"
            }]
        )
        print(f"Rad {index}: {completion.choices[0].message.content.strip()}")

ask_groq_rows(df)

Antal rader där area är Sverige i hela DataFrame: 65
Rad 237: Uppsala
Rad 274: Uppsala
Rad 406: Visby
Rad 440: Uppsala
Rad 441: Visby
Rad 532: Västerås
Rad 545: Västerås
Rad 615: Kalmar
Rad 618: Kalmar
Rad 623: Kalmar
Rad 624: Dalarna
Rad 626: Kalmar
Rad 632: Kalmar
Rad 736: Stockholm
Rad 1076: Stockholm
Rad 1197: Älvsborg
Rad 1205: Älvsborg
Rad 1227: Nyköping
Rad 1248: Älvsborg
Rad 1255: Älvsborg
Rad 1272: Älvsborg
Rad 1280: Älvsborg
Rad 1309: Stockholm
Rad 1369: Uppsala
Rad 1469: Stockholm
Rad 1825: Utländsk
Rad 1839: Stockholm
Rad 1843: Karlskrona
Rad 1868: Karlskrona
Rad 2014: Stockholm
Rad 2021: Stockholm
Rad 2024: Stockholm
Rad 2032: Stockholm
Rad 2034: Stockholm
Rad 2035: Stockholm
Rad 2036: Stockholm
Rad 2080: Stockholm
Rad 2081: Stockholm
Rad 2148: Utländsk
Rad 2161: Stockholm
Rad 2166: Viborg
Rad 2172: Stockholm
Rad 2196: Stockholm
Rad 2247: Stockholm
Rad 2248: Stockholm
Rad 2314: Stockholm
Rad 2321: Stockholm
Rad 2338: Stockholm
Rad 2368: Kungsträdgården
Rad 2370: Stockholm


In [11]:
def update_area_with_groq(df):
    subset = df[df['area'] == "Sverige"]
    
    for index, row in subset.iterrows():
        completion = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {"role": "system", "content": "Svara ENDAST med ett svenskt ortnamn eller 'Utländsk'. Svara aldrig 'Okänd', förklara aldrig, och använd inga tecken förutom själva namnet."},
                {"role": "user", "content": f"Ge mig orten i max två ord: {row['description']}"}
            ]
        )
        df.at[index, 'area'] = completion.choices[0].message.content.strip()

update_area_with_groq(df)

In [12]:
def check_replacement(df):
    # Räknar hur många rader som fortfarande har det generiska värdet "Sverige"
    remaining = len(df[df['area'] == "Sverige"])
    print(f"Antal rader kvar med 'Sverige': {remaining}")
    remainingokänd = len(df[df['area'] == "Okänd"])
    print(f"Antal rader kvar med 'Okänd': {remainingokänd}")
    
    # Visar de 10 första raderna för att se de nya ortnamnen
    print("\nFörsta 10 raderna i 'area' kolumnen:")
    print(df['area'].head(10000))

check_replacement(df)

Antal rader kvar med 'Sverige': 1
Antal rader kvar med 'Okänd': 302

Första 10 raderna i 'area' kolumnen:
0       Stralsund
1        Göteborg
2           Visby
3           Skåne
4            Vara
          ...    
2621    Gestilren
2622        Malmö
2623        Polen
2624        Okänd
2625        Malmö
Name: area, Length: 2626, dtype: object


In [13]:
def show_okanda(df):
    # Filtrerar rader där area är "Okänd" och visar de 10 första
    print(df[df['area'] == "Okänd"].head(10))

show_okanda(df)

   year                         title  \
18  500   Krig/konflikt i Ukraina 500   
25  500     Krig/konflikt i Polen 500   
27  500     Krig/konflikt i Polen 500   
28  500     Krig/konflikt i Polen 500   
33  516   Krig/konflikt i Estland 516   
34  517   Krig/konflikt i Ukraina 517   
38  587   Krig/konflikt i Ukraina 587   
39  593   Krig/konflikt i Ukraina 593   
42  600  Krig/konflikt i Jämtland 600   
43  600   Krig/konflikt i Sverige 600   

                                          description   area  \
18  På fältet stod fortfarande några svenska batal...  Okänd   
25  De svenska förlusterna under slaget uppgick ti...  Okänd   
27  Den sachsisk-polska armén förlorade omkring 2 ...  Okänd   
28  Den sachsiska armén bestod av sammanlagt 22 23...  Okänd   
33  Armén som ämnade angripa ryssarna skulle delas...  Okänd   
34  Under senare delen av slaget detacherades från...  Okänd   
38  Enligt Wennerholms beräkningar tillfångatogs m...  Okänd   
39  Enligt ryska relationer uppgick 

In [ ]:
def remove_ai_duplicates(df):
    # Filtrera endast de som är "Okänd"
    okand = df[df['area'] == "Okänd"]
    to_drop = []
    
    indices = okand.index.tolist()
    
    for i in range(len(indices)):
        if indices[i] in to_drop: continue
        
        for j in range(i + 1, len(indices)):
            if indices[j] in to_drop: continue
            
            desc1 = df.at[indices[i], 'description']
            desc2 = df.at[indices[j], 'description']
            
            prompt = f"Är dessa två texter samma händelse? Svara bara JA eller NEJ.\n1: {desc1}\n2: {desc2}"
            
            res = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}]
            ).choices[0].message.content.strip().upper()
            
            if "JA" in res:
                to_drop.append(indices[j])
                print(f"Tar bort dubblett: Index {indices[j]}")

    df.drop(to_drop, inplace=True)
    print(f"Klart! Tog bort {len(to_drop)} rader.")

remove_ai_duplicates(df)

Tar bort dubblett: Index 25
Tar bort dubblett: Index 33
Tar bort dubblett: Index 38
Tar bort dubblett: Index 73
Tar bort dubblett: Index 76
Tar bort dubblett: Index 78
Tar bort dubblett: Index 95
Tar bort dubblett: Index 102
Tar bort dubblett: Index 112
Tar bort dubblett: Index 126
Tar bort dubblett: Index 138
Tar bort dubblett: Index 161
Tar bort dubblett: Index 181
Tar bort dubblett: Index 661


In [15]:
import time
import re

ALLOWED_REGIONS = {"Svealand", "Götaland", "Norrland"}
BAD_ANSWERS = {"sverige", "okänd", "okand", "unknown", "", "utländsk", "utlandet", "none", "n/a"}

TITLE_PLACE_RE = re.compile(r"^\s*Krig/konflikt\s+i\s+(.+?)\s+\d+\s*$", re.IGNORECASE)

SYSTEM_PROMPT = (
    "Du är en expert på svensk historia och geografi. "
    "Svara ENDAST med ETT av följande, max två ord, inget annat:\n"
    " - ett svenskt ortnamn (stad, by, slott), t.ex. Sigtuna, Poltava\n"
    " - ett svenskt landskap, t.ex. Uppland, Västergötland, Skåne\n"
    " - ett UTLÄNDSKT land/ort om händelsen skedde utomlands, t.ex. Ukraina, Polen, Narva\n"
    " - en av storregionerna Svealand, Götaland, Norrland (sista utvägen i Sverige)\n"
    "FÖRBJUDET: svaren 'Sverige', 'Okänd', 'Utländsk', citattecken, punkter, förklaringar, flera alternativ. "
    "Använd webbsök för att läsa källan om det behövs."
)

FALLBACK_SYSTEM = (
    "Svara med EXAKT ett ord: antingen ett svenskt landskap (t.ex. Uppland, Skåne), "
    "en av regionerna Svealand/Götaland/Norrland, eller ett utländskt land (t.ex. Ryssland, Polen). "
    "Inget annat, inga tecken, ingen förklaring. Aldrig 'Sverige' eller 'Okänd'."
)

def _clean(ans: str) -> str:
    if not ans:
        return ""
    ans = re.sub(r"[\"'`*\.\,:;!\?]", "", ans).strip()
    ans = ans.split("\n")[0].strip()
    words = ans.split()
    if len(words) > 2:
        ans = " ".join(words[:2])
    return ans

def _is_valid(ans: str) -> bool:
    if not ans:
        return False
    if ans.lower() in BAD_ANSWERS:
        return False
    return True

def _from_title(title: str) -> str:
    if not title:
        return ""
    m = TITLE_PLACE_RE.match(title)
    if not m:
        return ""
    place = m.group(1).strip()
    if not _is_valid(place):
        return ""
    return place

def _ask(model: str, system: str, user: str) -> str:
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    return _clean(completion.choices[0].message.content or "")

def _ask_with_retry(model, system, user, max_attempts=3):
    for attempt in range(max_attempts):
        try:
            return _ask(model, system, user)
        except Exception as e:
            wait = 2 * (attempt + 1)
            print(f"    ! {type(e).__name__}: väntar {wait}s (försök {attempt+1}/{max_attempts})")
            time.sleep(wait)
    return ""

def resolve_okand_with_websearch(
    df,
    save_every=25,
    save_path=r"D:\SwedishGeoHistori\SwedishGeoHistory\scraper\krig_events_enriched.csv",
):
    subset_idx = df[df["area"] == "Okänd"].index.tolist()
    total = len(subset_idx)
    print(f"Antal 'Okänd' att lösa: {total}")
    if total == 0:
        return

    cache = {}
    saved_calls = 0
    from_title_hits = 0
    api_calls = 0

    for processed, idx in enumerate(subset_idx, start=1):
        row = df.loc[idx]

        title_place = _from_title(row["title"])
        if title_place:
            df.at[idx, "area"] = title_place
            from_title_hits += 1
            if processed % 25 == 0 or processed == total:
                print(f"  [{processed}/{total}] idx={idx} -> {title_place} (från titel)")
            continue

        key = (row["description"] or "")[:200]
        if key in cache:
            df.at[idx, "area"] = cache[key]
            saved_calls += 1
            continue

        user_prompt = (
            f"Titel: {row['title']}\n"
            f"År: {row['year']}\n"
            f"Källa: {row['source_url']}\n"
            f"Text: {(row['description'] or '')[:800]}\n\n"
            "Var ägde händelsen rum? Svensk ort/landskap om i Sverige, annars det utländska landet. Max två ord."
        )

        answer = _ask_with_retry("compound-beta", SYSTEM_PROMPT, user_prompt)
        api_calls += 1
        if not _is_valid(answer):
            fallback_user = (
                f"{row['title']} ({row['year']})\n"
                f"Källa: {row['source_url']}\n"
                f"{(row['description'] or '')[:500]}\n\n"
                "Vilket svenskt landskap/region eller utländskt land hör händelsen till?"
            )
            answer = _ask_with_retry("compound-beta", FALLBACK_SYSTEM, fallback_user)
            api_calls += 1
            if not _is_valid(answer):
                answer = "Svealand"

        cache[key] = answer
        df.at[idx, "area"] = answer

        if processed % 10 == 0 or processed == total:
            print(f"  [{processed}/{total}] idx={idx} -> {answer}")

        if processed % save_every == 0:
            df.to_csv(save_path, index=False, encoding="utf-8")
            print(f"  (checkpoint sparad till {save_path})")

    df.to_csv(save_path, index=False, encoding="utf-8")
    print(f"Klart. Sparade till {save_path}.")
    print(f"  Från titel: {from_title_hits}")
    print(f"  Cache-träffar: {saved_calls}")
    print(f"  API-anrop: {api_calls}")

resolve_okand_with_websearch(df)

Antal 'Okänd' att lösa: 302
    ! APIStatusError: väntar 2s (försök 1/3)
    ! APIStatusError: väntar 4s (försök 2/3)
    ! APIStatusError: väntar 6s (försök 3/3)
  [10/302] idx=43 -> Skåne
  [25/302] idx=97 -> Estland (från titel)
    ! RateLimitError: väntar 2s (försök 1/3)
  [30/302] idx=113 -> Svealand
    ! APIStatusError: väntar 2s (försök 1/3)
    ! RateLimitError: väntar 2s (försök 1/3)
    ! RateLimitError: väntar 2s (försök 1/3)
    ! APIStatusError: väntar 2s (försök 1/3)
    ! RateLimitError: väntar 2s (försök 1/3)
    ! APIStatusError: väntar 4s (försök 2/3)
    ! APIStatusError: väntar 2s (försök 1/3)
    ! RateLimitError: väntar 2s (försök 1/3)
    ! APIStatusError: väntar 4s (försök 2/3)
  (checkpoint sparad till D:\SwedishGeoHistori\SwedishGeoHistory\scraper\krig_events_enriched.csv)
  [80/302] idx=537 -> Dalarna
    ! RateLimitError: väntar 2s (försök 1/3)
    ! RateLimitError: väntar 2s (försök 1/3)
    ! RateLimitError: väntar 4s (försök 2/3)
    ! RateLimitError: v

In [16]:
def verify_areas(df):
    remaining_okand = len(df[df["area"] == "Okänd"])
    remaining_sverige = len(df[df["area"] == "Sverige"])
    print(f"Kvar 'Okänd':  {remaining_okand}")
    print(f"Kvar 'Sverige': {remaining_sverige}")
    print("\nTopp 15 area-värden:")
    print(df["area"].value_counts().head(15))
    long_answers = df[df["area"].str.split().str.len() > 2]
    if len(long_answers):
        print(f"\nVarning: {len(long_answers)} rader har mer än 2 ord (kan vara felaktiga AI-svar):")
        print(long_answers[["year", "title", "area"]].head(10))

verify_areas(df)

Kvar 'Okänd':  0
Kvar 'Sverige': 1

Topp 15 area-värden:
area
Stockholm      303
Svealand       173
Skåne           99
Malmö           91
Jämtland        90
Halmstad        71
Poltava         70
Vara            62
Göteborg        60
Lund            60
Uppsala         57
Stralsund       46
Helsingborg     43
Narva           39
Kalmar          36
Name: count, dtype: int64
